# JAX Vision-Range Curriculum

Train a fixed 50x50 AntByte task while shrinking the actor vision window from 51x51 to 3x3.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ant_byte_env import notebook_workflows as workflows

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status

/home/narf/miniconda3/envs/cool-antz/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


{'jax_already_imported': False,
 'jax_preallocate': 'false',
 'jax_memory_fraction': '0.35',
 'jax_allocator': 'platform',
 'memory_trimmed': True,
 'disk_free_gb': 28.14,
 'disk_used_percent': 75.7,
 'current_pid': 16831,
 'safe_cleanup_candidate_count': 0,
 'safe_cleanup_candidate_gb': 0.0,
 'top_memory_processes': [{'pid': 13069,
   'ppid': 8125,
   'rss_mb': 1004.7,
   'command': '/home/narf/.vscode-server/cli/servers/Stable-6928394f91b684055b873eecb8bc281365131f1c/server/node --dns-result-order=ipv4first /home/narf/.vscode-server/cli/servers/Stable-6928394f91b684055b873eecb8bc281365131f1c/server/out/bootstrap-fork --type=extensionHost --transformURIs --useHostProxy=false',
   'connection_file': None,
   'is_current_process': False,
   'is_notebook_kernel': False},
  {'pid': 13570,
   'ppid': 13069,
   'rss_mb': 790.4,
   'command': '/home/narf/.vscode-server/cli/servers/Stable-6928394f91b684055b873eecb8bc281365131f1c/server/node /home/narf/.vscode-server/extensions/ms-python.vscod

In [2]:
import jax

from ant_byte_env.experiments import load_experiment_config
from ant_byte_env.training.jax_mappo import runner as jax_runner

EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "vision_range_curriculum.json"
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "vision_range_curriculum"
experiment = load_experiment_config(EXPERIMENT_CONFIG)
TRAINING_ARGS = dict(experiment.args)
VISION_RADII = tuple(experiment.metadata["vision_radii"])
GLOBAL_UPDATE_CAP = int(experiment.metadata["global_update_cap"])
RENDER_MAX_FRAMES = int(experiment.metadata["render_max_frames"])
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
UPDATE_TIMESTEPS_PER_STAGE = int(TRAINING_ARGS["num_envs"]) * int(TRAINING_ARGS["num_steps"])
COMMON_ARGS = workflows.config_common_args(
    TRAINING_ARGS,
    exclude=workflows.VISION_RANGE_ARG_EXCLUDES,
)

{"backend": jax.default_backend(), "vision_radii": VISION_RADII}

{'backend': 'gpu', 'vision_radii': (20,)}

In [3]:
vision_result = workflows.run_vision_range_curriculum(
    vision_radii=VISION_RADII,
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    experiment_name=experiment.name,
    update_timesteps_per_stage=UPDATE_TIMESTEPS_PER_STAGE,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    max_render_frames=RENDER_MAX_FRAMES,
    tile_size=ROLLOUT_TILE_SIZE,
)
vision_result["final_checkpoint"]

/home/narf/miniconda3/envs/cool-antz/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training vision stage 1/1: 41x41


41x41: 0/5 updates |          | 00:00<? 2026-06-23 13:44:28.354490: E external/xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bw-input.4 = (f32[8000,64,13,13]{3,2,1,0}, u8[0]{0}) custom-call(%add.1308, %bitcast.337), window={size=3x3 stride=2x2 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_name="jit(<lambda>)/jit(main)/while/body/while/body/conv_general_dilated" source_file="/home/narf/Desktop/Facultad/RL/cool-antz-nuevo/cool-antz/src/ant_byte_env/training/jax_mappo/core.py" source_line=695}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-06-23 13:44:28.372351: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.017937951s
Trying algorithm eng0

PosixPath('/home/narf/Desktop/Facultad/RL/cool-antz-nuevo/cool-antz/runs/notebooks/vision_range_curriculum/41x41/checkpoints/model.pkl')

In [ ]:
from ant_byte_env.rendering import render_checkpoint

LONG_RENDER_MAX_FRAMES = 2_000
LONG_RENDER_STAGE_RADIUS = int(VISION_RADII[-1])
LONG_RENDER_STAGE_NAME = f"{workflows.vision_side(LONG_RENDER_STAGE_RADIUS)}x{workflows.vision_side(LONG_RENDER_STAGE_RADIUS)}"
LONG_RENDER_OUTPUT = RUN_DIR / "media" / f"vision_{LONG_RENDER_STAGE_NAME}_long_{LONG_RENDER_MAX_FRAMES}frames.gif"

LONG_RENDER_CHECKPOINT = None
if "vision_result" in globals():
    LONG_RENDER_CHECKPOINT = vision_result.get("final_checkpoint")
if LONG_RENDER_CHECKPOINT is None:
    LONG_RENDER_CHECKPOINT = RUN_DIR / LONG_RENDER_STAGE_NAME / "checkpoints" / "model.pkl"
LONG_RENDER_CHECKPOINT = Path(LONG_RENDER_CHECKPOINT)

render_checkpoint(
    LONG_RENDER_CHECKPOINT,
    LONG_RENDER_OUTPUT,
    backend="jax",
    seed_offset=workflows.NOTEBOOK_ROLLOUT_SEED_OFFSET + len(VISION_RADII) - 1,
    reuse_existing=False,
    max_frames=LONG_RENDER_MAX_FRAMES,
    tile_size=ROLLOUT_TILE_SIZE,
    policy_temperature=workflows.NOTEBOOK_ROLLOUT_POLICY_TEMPERATURE,
)

In [4]:
# import matplotlib.pyplot as plt

# metrics = vision_result["stage_metrics"]
# MEDIA_DIR = RUN_DIR / "media"
# MEDIA_DIR.mkdir(parents=True, exist_ok=True)
# updates = list(range(1, len(metrics) + 1))

# def metric_values(name):
#     return [float(row[name]) for row in metrics if name in row]

# fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
# for name in ("episode_return", "env_return"):
#     series = metric_values(name)
#     if series:
#         axes[0].plot(updates[: len(series)], series, label=name)
# for name in ("loss", "policy_loss", "value_loss"):
#     series = metric_values(name)
#     if series:
#         axes[1].plot(updates[: len(series)], series, label=name)
# for axis in axes:
#     axis.grid(True, alpha=0.3)
#     axis.legend()
# axes[0].set_ylabel("return")
# axes[1].set_ylabel("loss")
# axes[1].set_xlabel("curriculum update")
# fig.tight_layout()
# plot_path = MEDIA_DIR / "reward_loss.png"
# fig.savefig(plot_path, dpi=160)
# plot_path